# Week 8-A: Counterfactual Single-Passage Promotion Response Surface
## Draft-Aware Fidelity Routing (DAFiR) for Compressed RAG

This notebook measures the exact counterfactual marginal promotion utility $\Delta_i(q) = F_i(q) - F_0(q)$
for every candidate passage $i \in \{5, 6, 7, 8, 9, 10\}$ relative to the canonical semantic $k=4$ baseline ($4F + 6H$).

### Scientific Guarantees:
- Tests whether factual gains from ranks 5-10 can be unlocked without context bloat.
- Classifies each candidate into **HELP** (> +0.01), **NEUTRAL** ([-0.01, +0.01]), or **HURT** (< -0.01).
- Evaluates **Oracle-1**, **Oracle-Selective-1** ($\epsilon=0.01$), and **Greedy Oracle-2**.
- Strictly maintains paper-disjointness between FIT and INTERNAL-VAL.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
SARA_ROOT = REPO_ROOT / "Baselines" / "SARA-main"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src
if hasattr(src, "__path__") and str(SARA_ROOT / "src") not in src.__path__:
    src.__path__.append(str(SARA_ROOT / "src"))

from src.model.loader import load_sara_for_eval
from src.week8.semantic import SemanticScorer
from src.week8.data import load_config, load_week8_split
from src.week8.promotion_surface import run_single_promotion_surface
from src.week8.promotion_oracle import (
    compute_oracle_1,
    compute_oracle_selective_1,
    compute_best_rank_distribution,
    plot_oracle_results,
)

print("Environment and modules loaded successfully.")


In [2]:
# Load configuration and split data
cfg = load_config()
split = "internal_val"  # Change to 'fit' for full training surface
records, sfr_dict, manifest_df = load_week8_split(split, cfg)
print(f"Loaded {len(records)} queries for split '{split}'.")


Loaded 342 queries for split 'internal_val'.


In [3]:
# Compute semantic similarity rankings
scorer = SemanticScorer()
semantic_scores_by_qid = {}
for r in records:
    qid = str(r["question_id"])
    q_text = r["question"]
    passages = [p["text"] for p in r.get("passages", [])[:10]]
    sims = scorer.score_query_passages(q_text, passages).cpu().numpy()
    semantic_scores_by_qid[qid] = sims

print(f"Computed semantic rankings for {len(semantic_scores_by_qid)} queries.")


Computed semantic rankings for 342 queries.


In [4]:
# Load SARA generator
device = "cuda" if torch.cuda.is_available() else "cpu"
sara_ckpt_path = REPO_ROOT / cfg["sara"]["checkpoint_path"]
sara_model, sara_tokenizer = load_sara_for_eval(str(sara_ckpt_path), device=device)
print(f"Loaded SARA generator on {device}.")


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 120.29it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `

Loaded SARA generator on cuda.


In [5]:
# Run counterfactual single-passage promotion surface
cache_path = REPO_ROOT / "results/week8/surfaces" / f"promotion_surface_{split}.jsonl"
df_promotions = run_single_promotion_surface(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=cache_path,
    device=device,
)
print(f"Promotion surface completed with {len(df_promotions)} candidate evaluations.")


W8-A Promotion Surface:  16%|█▌        | 55/342 [19:54<1:43:54, 21.72s/query]


KeyboardInterrupt: 

In [ ]:
# Compute Oracle-1 and Oracle-Selective-1
df_o1, o1_summary = compute_oracle_1(df_promotions)
df_sel1, sel1_summary = compute_oracle_selective_1(df_promotions, epsilon=0.01)
df_ranks = compute_best_rank_distribution(df_o1)

print("=== Oracle-1 Summary ===")
for k, v in o1_summary.items():
    print(f"  {k}: {v}")

print("\n=== Oracle-Selective-1 Summary ===")
for k, v in sel1_summary.items():
    print(f"  {k}: {v}")


## Section 5: Evidence Potential and Generator Utilization Bottleneck Analysis

This section analyzes the **Three-Layer Conceptual Decomposition**:
$$\boxed{\textbf{Selection} \longrightarrow \textbf{Fidelity} \longrightarrow \textbf{Utilization}}$$

We classify each candidate promotion into the four-way taxonomy:
1. **UTILIZED-HELPFUL**: Gold evidence exposed and exploited ($\Delta F1 > 0.01$).
2. **UNDERUTILIZED**: Gold evidence exposed, but generator failed to exploit it ($\Delta F1 \le 0.01$, e.g. Q1812: answer stayed `"GloVe"`).
3. **TRUE-NEUTRAL**: Candidate adds no missing answer info; base draft was already sufficient (e.g. Q1811: $F1_0=1.0$, safe-stop).
4. **DISTRACTOR**: Promoted candidate harms answer quality ($\Delta F1 < -0.01$).


In [ ]:
from src.week8.evidence_enrichment import (
    load_qasper_gold_evidence_map,
    enrich_promotion_record,
)
from src.week8.evaluation import compute_evidence_utilization_metrics

# Load gold QASPER evidence paragraphs (100% matched across dataset)
evidence_map = load_qasper_gold_evidence_map()
print(f"Loaded {len(evidence_map)} ground-truth evidence mappings.")

# Enrich promotion records
enriched_records = []
for r in df_promotions.to_dict(orient="records"):
    en = enrich_promotion_record(r, evidence_map=evidence_map)
    enriched_records.append(en)

df_enriched = pd.DataFrame(enriched_records)
print(f"Enriched {len(df_enriched)} promotion candidate records.")


In [ ]:
# Compute Generator Utilization Metrics & Promotion Utilization Failure Rate (UFR)
util_metrics = compute_evidence_utilization_metrics(df_enriched.to_dict(orient="records"))

print("=== Evidence Potential & Generator Utilization Metrics ===")
print(f"Total Promotions Evaluated:              {util_metrics['total_promotions']}")
print(f"P(Answer Changed):                       {util_metrics['prob_answer_changed']:.4f}")
print(f"P(Answer Changed | Delta > 0):           {util_metrics['prob_answer_changed_given_help']:.4f}")
print(f"P(Answer Changed | Gold Evidence):       {util_metrics['prob_answer_changed_given_gold_evidence']:.4f}")
print(f"P(Delta > 0.01 | Gold Evidence):         {util_metrics['prob_help_given_gold_evidence']:.4f}")
print(f"P(Delta == 0 | Gold Evidence):           {util_metrics['prob_zero_delta_given_gold_evidence']:.4f}")
print(f"Promotion Utilization Failure Rate (UFR):{util_metrics['promotion_utilization_failure_rate']:.4f}")

tax_df = pd.DataFrame([
    {"Category": k, "Count": v["count"], "Pct": f"{v['pct']*100:.1f}%"}
    for k, v in util_metrics["taxonomy_distribution"].items()
])
print("\n=== Taxonomy Breakdown ===")
print(tax_df.to_string(index=False))


In [ ]:
# Case Studies: Safe-Stop (Q1811) vs Underutilized Evidence (Q1812)
q1811_rows = df_enriched[df_enriched["question_id"].astype(str) == "1811"]
if len(q1811_rows) > 0:
    print("=== Q1811 (Canonical Safe-Stop Query) ===")
    print("Question:", q1811_rows.iloc[0]["question"])
    print("Base Draft Answer:", q1811_rows.iloc[0]["base_prediction"])
    print("Base F1:", q1811_rows.iloc[0]["base_f1"])
    print("Promotions Delta F1:", q1811_rows["delta_f1"].tolist())
    print("Action Required: STOP (do nothing)")

print("\n" + "="*50 + "\n")

q1812_rows = df_enriched[df_enriched["question_id"].astype(str) == "1812"]
if len(q1812_rows) > 0:
    print("=== Q1812 (Underutilized Evidence Failure Mode) ===")
    print("Question:", q1812_rows.iloc[0]["question"])
    print("Base Draft Answer:", q1812_rows.iloc[0]["base_prediction"])
    print("Gold Answer:", q1812_rows.iloc[0]["gold_answers"])
    print("Candidate Ranks:", q1812_rows["candidate_passage_index"].tolist())
    print("Promotions Delta F1:", q1812_rows["delta_f1"].tolist())
    print("Taxonomy Labels:", q1812_rows["evidence_utilization_type"].tolist())
    print("Action Required: PROMOTE + DRAFT-AWARE REVISION (G1)")


In [ ]:
# Save Enriched Surface for W8-C Utility Modeling
enriched_path = REPO_ROOT / "results/week8/surfaces" / f"promotion_surface_{split}_enriched.jsonl"
df_enriched.to_json(enriched_path, orient="records", lines=True)
print(f"Saved enriched promotion surface to {enriched_path}")
